## Playbook 1 — disk is full

**Symptom:** "No space left on device", services failing to write, installs failing.

**1. Confirm and locate — bytes or inodes?**

```bash
df -h        # which filesystem is at 100%?
df -i        # or is it inodes? (lots of tiny files)
```

`Use%` at 100% is a bytes problem; `IUse%` at 100% on `df -i` is **inode exhaustion** — common on mail spools and build dirs.

**2. Find the big stuff (bytes):**

```bash
sudo du -h --max-depth=1 /var 2>/dev/null | sort -h | tail
sudo find /var -type f -size +100M -printf '%s %p\n' 2>/dev/null | sort -rn | head
```

(`ncdu /var` is faster if installed.)

**3. Check for deleted-but-held files — the classic gotcha:**

```bash
sudo lsof | grep deleted | head
```

A process holding a deleted file open keeps its space until it closes it — so `df` shows full while `du` doesn't (you saw this in module 07). **Fix: restart the offending service.**

**4. Inode exhaustion?** If `df -i` is full, hunt directories with the most files, then clear the culprit (PHP sessions, mail queues, package caches).

**5. Quick wins:**

```bash
sudo apt clean            # or: dnf clean all
sudo journalctl --vacuum-size=200M
sudo apt autoremove --purge   # old kernels
```
